# Introduction

👋 Welcome back to the practical sessions of the Generative AI course.

These tutorials complement the lectures by focusing on implementation. Each Friday session combines a pen-and-paper problem sheet, a short tour of the notebook, and time to work through the notebook.

## 🏫 Session Format

1. **Problem sheet (first hour, about 50 min)**: pen-and-paper exercises on the week's material, alone or in pairs. Written solutions are released on Scientia the following Monday.
2. **Break (10 min)**.
3. **Notebook tour (about 10 min)**: a few slides on how this notebook is organised and how it connects to the problem sheet.
4. **Notebook (until the end of the session)**: work through this notebook. Run the provided cells and change their parameters to see what happens. The exercises have code cells with `# TODO` comments and `None` placeholders for you to fill in. If you get stuck on an exercise, you can move on and come back to it later: until an exercise passes its check, the later sections fall back on PyTorch's implementation (or skip the step that needs it). Part 4 is optional, for when you have time left. Raise your hand if you need help.

**Smart AI Use**: You have access to Imperial's AI tools via [dAIsy](https://www.imperial.ac.uk/admin-services/ict/training-and-resources/daisy/). Letting an AI write the exercise code for you defeats the purpose of the exercise. Treat it as a copilot instead: ask it about unfamiliar APIs, or use it to help debug errors you cannot solve on your own.

## 💻 Computational Environment

This notebook needs only PyTorch and matplotlib, downloads nothing, and runs in about a minute or less on a laptop CPU (most of the time goes into training a tiny GPT in Section 3.1), so you do not need a GPU today. It also runs on Google Colab and on Apple-silicon Macs (the setup cell below picks the MPS backend automatically). If you already have a GPU set up from the logistics lecture (GPU Dojo, a lab workstation or the department cluster), you are welcome to use it; remember to stop a GPU Dojo VM when you finish, because its GPU-hour budget is used whenever it is running.

If `import torch` fails in the setup cell, run `%pip install torch matplotlib` in a new cell. On lab machines and the cluster, install into a virtual environment on `/vol/bitbucket/$USER` rather than your home directory, which is small.

## 🛠️ Jupyter Notebook Workflow

- Run a cell with **Shift + Enter**.
- Run the cells **in order, from the top**. The setup cell below defines `device` and `SEED`, which every Part uses. Exercise 1.1 defines `attention`, which the rest of the notebook calls: it is your `causal_attention` once that passes its check, and PyTorch's attention otherwise. In the same way, Part 3 uses your multi-head attention from Exercise 1.2 and your sampling function from Exercise 3.2 once they pass their checks. Section 1.4 and Part 2 need only the setup cell. Part 4 fine-tunes the model trained in Section 3.1.

Run the cell below to check your environment.

In [ ]:
import os
import shutil
import subprocess

import torch

print(f"PyTorch version: {torch.__version__}")

# The name and memory of each NVIDIA GPU (like `!nvidia-smi`, but short), without failing on machines without an NVIDIA driver
if shutil.which("nvidia-smi"):
    query = ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]
    print(subprocess.run(query, capture_output=True, text=True).stdout)
else:
    print("nvidia-smi not found: this machine has no NVIDIA driver.")

# Device: CUDA, then Apple MPS, then CPU. Set the environment variable T2_DEVICE (e.g. "cpu") to override.
if os.environ.get("T2_DEVICE"):
    device = torch.device(os.environ["T2_DEVICE"])
elif torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

if device.type == "cuda":
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
elif device.type == "mps":
    print("✅ Using the Apple-silicon GPU (MPS).")
else:
    print("⚠️ Using the CPU. This notebook still runs, but on a GPU machine it means PyTorch cannot see the GPU.")

# Full float32 precision on NVIDIA GPUs, for the exact comparisons in the exercises
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

SEED = 0
torch.manual_seed(SEED)  # seeds the random number generators of all devices
print(f"Global seed: {SEED}")

# Tutorial 2: Transformers

In this notebook we implement the main pieces of a decoder-only Transformer such as GPT-2, following the lecture and the problem sheet. Part 1 builds masked multi-head self-attention, counts the parameters of GPT-2 small and implements rotary position embeddings (RoPE). Part 2 computes attention block by block, the idea behind FlashAttention. Part 3 trains a tiny character-level GPT, samples from it with temperature, top-k and top-p, and speeds up generation with a KV cache. Part 4 (optional) fine-tunes the tiny GPT with LoRA.

By the end of this session you will be able to:
- implement $\text{softmax}(QK^\top/\sqrt{d_k})\,V$ with a causal mask, split it into heads, and reproduce PyTorch's attention;
- count the parameters of GPT-2 small and check your answers to the problem sheet;
- apply RoPE and check that the attention score depends only on the distance between two positions;
- compute exact attention over blocks of keys with a running maximum and a running denominator;
- sample the next token with temperature, top-k and top-p, and generate text with a KV cache;
- add a LoRA adapter to a linear layer and train only the adapter.

The reading list is on the lecture's "Reading Material and Reference" slide (p.97).

## Part 1: Masked Multi-Head Attention

- **Causal attention**: scores, mask, softmax and output for one head, checked on the problem sheet's Q2.
- **Heads**: split $Q$, $K$, $V$ into heads, attend in every head at once, and merge the heads.
- **Counting**: build GPT-2 small without allocating memory and count its parameters (the problem sheet's Q3).
- **RoPE**: rotate queries and keys by their positions, checked on the problem sheet's Q4.

### **1.1 Exercise: Causal Attention by Hand**

In this exercise you implement masked (causal) self-attention for one head and check it on the problem sheet's Q2.

**Scenario**:
Tokens are rows: $Q, K \in \mathbb{R}^{L \times d_k}$, $V \in \mathbb{R}^{L \times d_v}$, and

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V,$$

with the softmax taken over each row. For causal attention, the scores above the diagonal (key $j$ later than query $i$) are set to $-\infty$ before the softmax, so no token attends to later tokens. Positions are counted from 0.

**Requirements**: complete `causal_attention(Q, K, V)`, which returns the output $O$ and the weights $A$. It must accept any number of leading batch dimensions (later, `Q` has shape `(B, h, L, d_k)`), so use only the last two dimensions.

1. **Scores (TODO 1)**: $S = QK^\top / \sqrt{d_k}$, of shape `(..., L, L)`.
2. **Mask (TODO 2)**: set the entries of $S$ above the diagonal to $-\infty$.
3. **Weights and output (TODO 3)**: $A$, the row-wise softmax of $S$, and $O = AV$.

The verification code checks your $A$ and $O$ on the matrices of the problem sheet's Q2, and compares your function with `F.scaled_dot_product_attention(..., is_causal=True)` on random input.

**Hint**: useful are `K.transpose(-2, -1)`, `torch.triu(..., diagonal=1)`, `masked_fill` and `torch.softmax(..., dim=-1)`.

In [ ]:
import math

import torch.nn.functional as F


def causal_attention(Q, K, V):
    """Masked self-attention for Q, K (..., L, d_k) and V (..., L, d_v). Returns O (..., L, d_v) and A (..., L, L)."""
    L, d_k = Q.shape[-2], Q.shape[-1]

    # TODO 1: the scaled scores S = Q K^T / sqrt(d_k), shape (..., L, L)
    S = None

    # TODO 2: set the entries of S above the diagonal (key j > query i) to -inf
    pass

    # TODO 3: A = the row-wise softmax of S, and O = A V
    A = O = None
    return O, A


# --- Verification Code (Implemented) ---
print("-" * 30)
# The problem sheet's Q2: one head, three tokens (one per row), d_k = 4, so sqrt(d_k) = 2
Q2_Q = torch.tensor([[0.0, 2, 2, 2], [1, 1, 0, 0], [0, 0, 1, 1]])
Q2_K = torch.tensor([[1.0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 1]])
Q2_V = torch.tensor([[2.0, 0], [0, 2], [1, 1]])
e = math.e
expected_A = torch.tensor([[1, 0, 0], [1 / 2, 1 / 2, 0], [1 / (2 + e), 1 / (2 + e), e / (2 + e)]])
expected_O = torch.tensor([[2.0, 0], [1, 1], [1, 1]])

O, A = causal_attention(Q2_Q.to(device), Q2_K.to(device), Q2_V.to(device))
if O is None or A is None:
    print("Please complete the TODO sections to run the verification.")
    causal_attention_ok = False
else:
    O, A = O.cpu(), A.cpu()
    # atol=1e-6: float32 keeps about 7 significant digits; a wrong scale or mask changes some weights by 0.1 or more
    A_match = A.shape == expected_A.shape and torch.allclose(A, expected_A, atol=1e-6)
    O_match = O.shape == expected_O.shape and torch.allclose(O, expected_O, atol=1e-6)
    print(f"Q2 weights A (rows: queries, columns: keys):\n{A.numpy().round(4)}\nexpected:\n{expected_A.numpy().round(4)}")
    print(f"match: {A_match}\n")
    print(f"Q2 output O = A V:\n{O.numpy().round(4)}\nexpected:\n{expected_O.numpy()}")
    print(f"match: {O_match}\n")

    torch.manual_seed(SEED)
    Q, K, V = (torch.randn(2, 3, 6, 8, device=device) for _ in range(3))  # (B, h, L, d_k): 2 sequences, 3 heads, 6 tokens
    mine, _ = causal_attention(Q, K, V)
    ref = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    # atol=1e-5 allows for rounding differences between our matrix products and PyTorch's attention kernels
    sdpa_match = mine.shape == ref.shape and torch.allclose(mine, ref, atol=1e-5)
    print(f"random input: yours {tuple(mine.shape)}, F.scaled_dot_product_attention {tuple(ref.shape)}, match: {sdpa_match}")
    if mine.shape == ref.shape and not sdpa_match:
        print(f"    max absolute difference: {(mine - ref).abs().max().item():.3g}")
    causal_attention_ok = A_match and O_match and sdpa_match

# Q2(D): without the mask, the first token (row 0) would put most of its weight on the future tokens
unmasked = torch.softmax(Q2_Q @ Q2_K.T / 2, dim=-1)[0]
print(f"\nwithout the mask, row 0 would be {unmasked.numpy().round(4)} = [1, e, e^2] / (1 + e + e^2)")


def attention(Q, K, V, causal=True):
    """The attention used in the rest of the notebook: your causal_attention once it passes the check above,
    PyTorch's F.scaled_dot_product_attention otherwise (and always when causal=False)."""
    if causal and causal_attention_ok:
        return causal_attention(Q, K, V)[0]
    return F.scaled_dot_product_attention(Q, K, V, is_causal=causal)


print("\nThe rest of the notebook uses " + ("your causal_attention." if causal_attention_ok else
      "PyTorch's attention until your causal_attention passes this check (re-run this cell when it does)."))

### **1.2 Exercise: Splitting into Heads**

In this exercise you implement multi-head self-attention for a batch of sequences, following the reshape recipe of lecture p.39.

**Scenario**:
The input `x` has shape `(B, L, D_k)`. The layers `self.W_q`, `self.W_k`, `self.W_v` and `self.W_o` are provided as `nn.Linear(D_k, D_k)` (with biases, as in GPT-2). With $h$ heads, $D_k = h \cdot D_h$, and each head attends with its own $D_h$ columns of $Q$, $K$ and $V$:

| Step | Shape |
|---|---|
| $Q$, $K$, $V$ | `(B, L, D_k)` |
| reshape, then transpose | `(B, L, h, D_h)`, then `(B, h, L, D_h)` |
| attention in every head | `(B, h, L, D_h)` |
| transpose back, then reshape (concatenates the heads) | `(B, L, h, D_h)`, then `(B, L, D_k)` |
| $W_o$ | `(B, L, D_k)` |

**Requirements**: complete `forward(x, causal=True)`.

1. **Projections (TODO 1)**: `Q`, `K` and `V`.
2. **Split (TODO 2)**: each into heads, `(B, h, L, D_h)`.
3. **Attention (TODO 3)**: `attention(Q, K, V, causal)` from Exercise 1.1, which treats `(B, h)` as batch dimensions.
4. **Merge (TODO 4)**: back to `Z` of shape `(B, L, D_k)`; the provided code applies $W_o$.

The verification code copies the weights into PyTorch's multi-head attention (`F.multi_head_attention_forward`, which `nn.MultiheadAttention` uses) and compares the outputs for $B = 2$, $L = 7$, $D_k = 32$ and $h = 4$.

**Hint**: use `view` and `transpose(1, 2)`. After a transpose the memory is no longer in row order, so merge the heads with `reshape` (`view` raises an error).

In [ ]:
import torch.nn as nn


class MultiHeadSelfAttention(nn.Module):
    """Masked multi-head self-attention (lecture p.39), rows are tokens: x (B, L, D_k) -> (B, L, D_k)."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "D_k must be a multiple of the number of heads"
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, causal=True):
        B, L, D_k = x.shape
        h = self.n_heads
        D_h = D_k // h

        # TODO 1: Q = x W_q, K = x W_k, V = x W_v (with their biases), each (B, L, D_k)
        Q = K = V = None

        # TODO 2: split into heads: reshape each to (B, L, h, D_h), then transpose to (B, h, L, D_h)
        pass

        # TODO 3: attention in every head at once with attention(Q, K, V, causal) from Exercise 1.1 -> (B, h, L, D_h)
        O = None

        # TODO 4: merge the heads: transpose back to (B, L, h, D_h), then reshape to Z of shape (B, L, D_k)
        Z = None

        if Z is None:  # TODO not done yet
            return None
        return self.W_o(Z)


class ReferenceMultiHeadSelfAttention(MultiHeadSelfAttention):
    """The same layer computed by PyTorch's multi-head attention (used while your forward does not match yet).

    It applies the four layers itself and gives PyTorch identity projections, so that it also works when a
    layer is replaced, e.g. by the LoRA layer of Part 4.
    """

    def forward(self, x, causal=True):
        B, L, D_k = x.shape
        mask = nn.Transformer.generate_square_subsequent_mask(L).to(x.device) if causal else None  # -inf above the diagonal
        Q, K, V = (layer(x).transpose(0, 1) for layer in (self.W_q, self.W_k, self.W_v))  # (L, B, D_k), PyTorch's layout here
        eye, zeros = torch.eye(D_k, device=x.device), torch.zeros(D_k, device=x.device)
        Z, _ = F.multi_head_attention_forward(
            Q, K, V, D_k, self.n_heads, None, zeros.repeat(3), None, None, False, 0.0, eye, zeros,
            need_weights=False, attn_mask=mask, use_separate_proj_weight=True,
            q_proj_weight=eye, k_proj_weight=eye, v_proj_weight=eye)
        return self.W_o(Z.transpose(0, 1))


# --- Verification Code (Implemented) ---
print("-" * 30)
torch.manual_seed(SEED)
mine_layer = MultiHeadSelfAttention(32, n_heads=4).to(device)
ref_layer = ReferenceMultiHeadSelfAttention(32, n_heads=4).to(device)
ref_layer.load_state_dict(mine_layer.state_dict())  # the same weights
x = torch.randn(2, 7, 32, device=device)  # B = 2 sequences of L = 7 tokens, D_k = 32
with torch.no_grad():
    mine = mine_layer(x)
    ref = ref_layer(x)
if mine is None:
    print("Please complete the TODO sections to run the verification.")
    mhsa_ok = False
else:
    # atol=1e-5 allows for rounding differences between our computation and PyTorch's kernels;
    # a real mistake (e.g. splitting the heads in the wrong order) gives errors of 0.01 or more
    mhsa_ok = mine.shape == ref.shape and torch.allclose(mine, ref, atol=1e-5)
    print(f"yours {tuple(mine.shape)}, PyTorch's multi-head attention {tuple(ref.shape)}, match: {mhsa_ok}")
    if mine.shape == ref.shape and not mhsa_ok:
        print(f"    max absolute difference: {(mine - ref).abs().max().item():.3g}")

### **1.3 Counting GPT-2 Small**

The cell below defines the decoder-only model used in the rest of the notebook, shaped like GPT-2: a token embedding plus a learned position embedding, $N$ Transformer blocks, a final LayerNorm, and a linear output layer with one logit per vocabulary token. Each block has masked multi-head self-attention (your `MultiHeadSelfAttention`) and an MLP $D \to 4D \to D$ with GELU, each with a residual connection and, as in GPT-2, a LayerNorm in front.

It then builds GPT-2 small with the assumptions of the problem sheet's Q3 on PyTorch's `meta` device, which records the parameter shapes without allocating memory, and counts the parameters. Compare the numbers with your answers.

In [ ]:
import torch.nn as nn


class Block(nn.Module):
    """A GPT-2 Transformer block: x + attention(LayerNorm(x)), then x + MLP(LayerNorm(x))."""

    def __init__(self, d_model, n_heads, attention_class=None):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = (attention_class or MultiHeadSelfAttention)(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    """Decoder-only Transformer like GPT-2: token ids (B, L) -> logits (B, L, vocab_size)."""

    def __init__(self, vocab_size, d_model, n_heads, n_blocks, context_length, attention_class=None):
        super().__init__()
        self.context_length = context_length
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context_length, d_model)  # learned position embedding, positions 0 .. context_length - 1
        self.blocks = nn.ModuleList([Block(d_model, n_heads, attention_class) for _ in range(n_blocks)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying: the output layer reuses the token-embedding matrix
        nn.init.normal_(self.tok_emb.weight, std=0.02)  # small initial embeddings, as in GPT-2
        nn.init.normal_(self.pos_emb.weight, std=0.02)

    def forward(self, idx):
        positions = torch.arange(idx.shape[1], device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(positions)  # (B, L, D)
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))


def count_parameters(module):
    """Number of parameter values; a parameter shared by two layers (weight tying) is counted once."""
    return sum(p.numel() for p in module.parameters())


with torch.device("meta"):  # shapes only, no memory
    gpt2 = GPT(vocab_size=50257, d_model=768, n_heads=12, n_blocks=12, context_length=1024)

D, n_blocks, N_ctx = 768, len(gpt2.blocks), 1024
block = gpt2.blocks[0]
print(f"head dimension D_h = {D} / {block.attn.n_heads} = {D // block.attn.n_heads}")
print(f"attention W_q, W_k, W_v, W_o: {count_parameters(block.attn):>12,}  (4D^2 + 4D)")
print(f"MLP 768 -> 3072 -> 768:       {count_parameters(block.mlp):>12,}  (8D^2 + 5D)")
print(f"two LayerNorms:               {count_parameters(block.ln1) + count_parameters(block.ln2):>12,}  (4D)")
print(f"one block:                    {count_parameters(block):>12,}  (Q3: 7,087,872), match: {count_parameters(block) == 7_087_872}\n")

print(f"token embedding 50,257 x 768: {count_parameters(gpt2.tok_emb):>12,}")
print(f"position embedding 1024 x 768:{count_parameters(gpt2.pos_emb):>12,}")
print(f"12 blocks:                    {count_parameters(gpt2.blocks):>12,}")
print(f"final LayerNorm:              {count_parameters(gpt2.ln_f):>12,}")
print(f"output layer (tied):          {0:>12,}")
total = count_parameters(gpt2)
print(f"total:                        {total:>12,}  (Q3: 124,439,808), match: {total == 124_439_808}\n")

# Q3(A) and Q3(C): memory, at 2 bytes per value
scores = block.attn.n_heads * N_ctx**2  # one N x N score matrix per head, in one layer
kv_values = 2 * n_blocks * N_ctx * D  # K and V: D values per token per block
print(f"score entries per layer for N = {N_ctx}: {scores:,} = {2 * scores:,} bytes (about {2 * scores / 1e6:.0f} MB, more than the 20 MB of SRAM)")
print(f"KV cache for N = {N_ctx}: {kv_values:,} values = {2 * kv_values:,} bytes (about {2 * kv_values / 1e6:.0f} MB), "
      f"match: {2 * kv_values == 37_748_736}")

### **1.4 Exercise: Rotary Position Embedding**

RoPE encodes positions by rotating queries and keys (lecture p.36). In two dimensions, the vector $x_m$ at position $m$ becomes $f(x_m, m) = R_m x_m$, where $R_m$ rotates by the angle $m\theta$; then $f(x_m, m) \cdot f(x_n, n) = x_m^\top R_{n-m}\, x_n$ depends on the positions only through $n - m$. For $d > 2$, the entries form pairs $(x_{2i}, x_{2i+1})$, $i = 0, \dots, d/2 - 1$, and pair $i$ is rotated by $m\theta_i$ with $\theta_i = 10000^{-2i/d}$ (the slide gives no formula for $\theta_i$; this is the usual choice).

**Requirements**: complete `apply_rope(x, positions, thetas)` for `x` of shape `(..., L, d)` (one row per token), integer `positions` of shape `(L,)` and `thetas` of shape `(d/2,)`.

1. **Angles (TODO 1)**: `angles[m, i]` $= \text{positions}[m] \cdot \theta_i$, of shape `(L, d/2)`.
2. **Pairs (TODO 2)**: `x1` $= (x_0, x_2, \dots)$ and `x2` $= (x_1, x_3, \dots)$, each `(..., L, d/2)`.
3. **Rotation (TODO 3)**: rotate every pair, $(x_1 \cos - x_2 \sin,\; x_1 \sin + x_2 \cos)$, and interleave the results back into $(x'_0, x'_1, x'_2, \dots)$.

The verification code checks the problem sheet's Q4 ($d = 2$, $\theta = \pi/2$), the pairing on a $d = 4$ example, and, with random $d = 64$ vectors, that shifting both positions by the same amount leaves the score unchanged. It runs on the CPU in double precision (which MPS does not support).

**Hint**: `x[..., 0::2]` takes every second entry; `torch.stack([a, b], dim=-1).flatten(-2)` interleaves `a` and `b`.

In [ ]:
import math


def apply_rope(x, positions, thetas=None):
    """Rotate the pairs (x_2i, x_2i+1) of every row of x (..., L, d) by positions[m] * thetas[i]."""
    d = x.shape[-1]
    if thetas is None:
        thetas = 10000 ** (-torch.arange(0, d, 2, dtype=x.dtype, device=x.device) / d)  # theta_i = 10000^(-2i/d)
    positions = positions.to(x.dtype)

    # TODO 1: angles[m, i] = positions[m] * thetas[i], shape (L, d/2)
    angles = None

    # TODO 2: x1 = the first entries (x_0, x_2, ...) and x2 = the second entries (x_1, x_3, ...) of the pairs, each (..., L, d/2)
    x1 = x2 = None

    if angles is None or x1 is None:  # TODO 1 or TODO 2 not done yet
        return None
    cos, sin = torch.cos(angles), torch.sin(angles)

    # TODO 3: rotate every pair, (x1 cos - x2 sin, x1 sin + x2 cos), and interleave the results back to shape (..., L, d)
    return None


# --- Verification Code (Implemented) ---
print("-" * 30)
f64 = torch.float64


def rope_score(q, m, k, n, thetas=None):
    """f(q, m) . f(k, n) for single vectors q, k at positions m, n."""
    fq = apply_rope(q[None], torch.tensor([m]), thetas)
    fk = apply_rope(k[None], torch.tensor([n]), thetas)
    return None if fq is None or fk is None else (fq[0] @ fk[0]).item()


q = torch.tensor([1.0, 0.0], dtype=f64)
k = torch.tensor([1.0, 1.0], dtype=f64)
half_pi = torch.tensor([math.pi / 2], dtype=f64)  # the problem sheet's theta = pi/2
if apply_rope(q[None], torch.tensor([1]), half_pi) is None:
    print("Please complete the TODO sections to run the verification.")
else:
    # atol=1e-9: cos(pi/2) is about 6e-17 in double precision, not exactly 0
    Rq = apply_rope(q[None], torch.tensor([1]), half_pi)[0]
    Rk = apply_rope(k[None], torch.tensor([2]), half_pi)[0]
    print(f"Q4: R_1 q = {Rq.numpy().round(6) + 0.0}, expected [0, 1];  R_2 k = {Rk.numpy().round(6) + 0.0}, expected [-1, -1]")
    print(f"    match: {torch.allclose(Rq, torch.tensor([0.0, 1], dtype=f64), atol=1e-9) and torch.allclose(Rk, torch.tensor([-1.0, -1], dtype=f64), atol=1e-9)}")
    for m, n in [(1, 2), (2, 3)]:
        s = rope_score(q, m, k, n, half_pi)
        print(f"Q4: f(q, {m}) . f(k, {n}) = {s:.6f}, expected -1, match: {abs(s + 1) < 1e-9}")

    x = torch.tensor([[1.0, 2, 3, 4]], dtype=f64)
    pairs = apply_rope(x, torch.tensor([1]), torch.tensor([math.pi / 2, math.pi], dtype=f64))[0]
    print(f"pairs (x0, x1), (x2, x3) rotated by pi/2 and pi: {pairs.numpy().round(6) + 0.0}, expected [-2, 1, -3, -4], "
          f"match: {torch.allclose(pairs, torch.tensor([-2.0, 1, -3, -4], dtype=f64), atol=1e-9)}")

    torch.manual_seed(SEED)
    q64, k64 = torch.randn(64, dtype=f64), torch.randn(64, dtype=f64)
    scores = [rope_score(q64, 3 + shift, k64, 7 + shift) for shift in (0, 10, 100)]
    print(f"d = 64: f(q, m) . f(k, n) for (m, n) = (3, 7), (13, 17), (103, 107): {[round(s, 6) for s in scores]}, "
          f"match: {max(scores) - min(scores) < 1e-9}")
    print(f"        for comparison, (m, n) = (3, 8): {rope_score(q64, 3, k64, 8):.6f}")

## Part 2: Attention by Blocks (FlashAttention idea)

Standard attention stores the $N \times N$ matrices $S$ and $A$ in the GPU's large but slow memory (HBM). FlashAttention instead loads $K$ and $V$ block by block into the small, fast on-chip memory (SRAM) and never stores $S$ or $A$. The catch is the softmax denominator, which needs a whole row of scores: the output is rescaled whenever a new block arrives.

- **Online softmax over blocks**: implement this rescaling for a batch of queries, check it on the problem sheet's Q5, and compare it with ordinary attention for several block sizes.

### **2.1 Exercise: Online Softmax over Blocks**

In this exercise you compute $\text{softmax}(QK^\top/\sqrt{d})\,V$ (no mask) one block of keys at a time.

**Scenario**:
We use the lecture's rescaling (p.51-52) together with the running maximum $m$ of the online softmax (p.49), which keeps $\exp$ from overflowing (in float32, $e^{89}$ is already infinite). For every block $j$ of keys and values, for all queries at once (one row per query):

$$S^{(j)} = \frac{Q (K^{(j)})^\top}{\sqrt{d}}, \qquad m^{(j)} = \max\!\big(m^{(j-1)}, \text{row max of } S^{(j)}\big), \qquad A^{(j)} = \exp\!\big(S^{(j)} - m^{(j)}\big),$$

$$l^{(j)} = e^{m^{(j-1)} - m^{(j)}}\, l^{(j-1)} + \text{row sums of } A^{(j)}, \qquad O^{(j)} = \frac{e^{m^{(j-1)} - m^{(j)}}\, l^{(j-1)}}{l^{(j)}}\, O^{(j-1)} + \frac{A^{(j)} V^{(j)}}{l^{(j)}},$$

starting from $m^{(0)} = -\infty$, $l^{(0)} = 0$ and $O^{(0)} = 0$. With $m = 0$ throughout, these are the lecture's formulas; the provided code records $l\,e^{m}$, which is the lecture's $l^{(j)}$.

**Requirements**: complete the loop body of `blockwise_attention(Q, K, V, block_size)` for `Q` of shape `(N_q, d)`, `K` of shape `(N, d)` and `V` of shape `(N, d_v)`. The running `m` and `l` (both `(N_q, 1)`), `O` (`(N_q, d_v)`) and the blocks `K_j`, `V_j` are provided.

1. **Scores (TODO 1)**: `S_j`, of shape `(N_q, block_size)`.
2. **Maximum and denominator (TODO 2)**: `m_new`, `A_j` and `l_new`.
3. **Output (TODO 3)**: the new `O`; then `m, l = m_new, l_new`.

The verification code runs the problem sheet's Q5 (one query, two blocks of two keys) and compares your function with ordinary attention on random input for several block sizes, including one that does not divide $N$, and for scores so large that $\exp$ alone would overflow.

**Hint**: useful are `amax(dim=1, keepdim=True)`, `torch.maximum` and `sum(dim=1, keepdim=True)`; an `(N_q, 1)` tensor multiplies each row of an `(N_q, d_v)` tensor by its own number.

In [ ]:
import math

import torch.nn.functional as F


def blockwise_attention(Q, K, V, block_size):
    """softmax(Q K^T / sqrt(d)) V for Q (N_q, d), K (N, d), V (N, d_v), one block of keys at a time.
    Returns O (N_q, d_v) and, after every block, the lecture's l^(j) and O^(j)."""
    N_q, d = Q.shape
    m = torch.full((N_q, 1), float("-inf"), device=Q.device, dtype=Q.dtype)  # running maximum of each query's scores
    l = torch.zeros(N_q, 1, device=Q.device, dtype=Q.dtype)  # running denominator (relative to e^m)
    O = torch.zeros(N_q, V.shape[1], device=Q.device, dtype=Q.dtype)  # output so far, divided by the current l
    history = []
    for start in range(0, K.shape[0], block_size):
        K_j, V_j = K[start : start + block_size], V[start : start + block_size]  # one block, only this one is "loaded"

        # TODO 1: the scores of this block, S_j = Q K_j^T / sqrt(d), shape (N_q, block)
        S_j = None

        # TODO 2: m_new = max(m, row maximum of S_j); A_j = exp(S_j - m_new); l_new = e^(m - m_new) l + row sums of A_j
        m_new = A_j = l_new = None

        if S_j is None or l_new is None:  # TODO 1 or TODO 2 not done yet
            return None, None

        # TODO 3: O = (e^(m - m_new) l / l_new) O + A_j V_j / l_new; then m, l = m_new, l_new
        pass

        history.append((l * torch.exp(m), O.clone()))  # the lecture's l^(j) (which has no running maximum) and O^(j)
    return O, history


# --- Verification Code (Implemented) ---
print("-" * 30)
# The problem sheet's Q5: one query; with d = 1, Q = [[1]] and K = the scaled scores give S = [0, log 3, log 2, log 2]
Q5_Q = torch.tensor([[1.0]], device=device)
Q5_K = torch.tensor([[0.0], [math.log(3)], [math.log(2)], [math.log(2)]], device=device)
Q5_V = torch.tensor([[4.0], [0.0], [2.0], [6.0]], device=device)
O, history = blockwise_attention(Q5_Q, Q5_K, Q5_V, block_size=2)
if O is None:
    print("Please complete the TODO sections to run the verification.")
else:
    # atol=1e-5: float32 rounding of log 2 and log 3; a wrong rescaling gives errors of 0.1 or more
    for j, ((l_j, O_j), (l_exp, O_exp)) in enumerate(zip(history, [(4.0, 1.0), (8.0, 2.5)]), start=1):
        ok = abs(l_j.item() - l_exp) < 1e-5 and abs(O_j.item() - O_exp) < 1e-5
        print(f"Q5 after block {j}: l^({j}) = {l_j.item():.4f}, O^({j}) = {O_j.item():.4f}, "
              f"expected {l_exp:g} and {O_exp:g}, match: {ok}")

    torch.manual_seed(SEED)
    Q = torch.randn(64, 32, device=device)
    K = torch.randn(256, 32, device=device)
    V = torch.randn(256, 16, device=device)
    for block_size, scale in [(1, 1), (16, 1), (50, 1), (256, 1), (16, 30)]:
        Qs = scale * Q  # scale 30 gives scores above 100: exp alone would overflow
        mine, _ = blockwise_attention(Qs, K, V, block_size)
        # ordinary (unmasked) attention, in double precision on the CPU
        S = Qs.cpu().double() @ K.cpu().double().T / math.sqrt(Q.shape[1])
        ref = (torch.softmax(S, dim=-1) @ V.cpu().double()).float()
        # atol=1e-4 allows for float32 rounding: with scores above 100, the scores themselves are only accurate
        # to about 1e-5, and the weights inherit this; a wrong rescaling gives errors of 0.01 or more
        match = mine.shape == ref.shape and torch.allclose(mine.cpu(), ref, atol=1e-4)
        print(f"block size {block_size:3d}, largest score {S.max().item():6.1f}, exp overflows: {bool(torch.exp(S.float()).isinf().any())!s:5s}, "
              f"largest score block 64 x {min(block_size, 256):3d} (full: 64 x 256), match: {match}")

In each step only a `(queries x block size)` block of scores exists, never the full `64 x 256` matrix, yet the result equals ordinary attention. On a GPU, FlashAttention does this inside one kernel, with each block in SRAM; this Python loop shows the arithmetic but is not faster.

## Part 3: A Tiny GPT and Decoding

- **Training**: train a small GPT-like model on characters of a public-domain text, using your attention from Part 1.
- **Sampling**: temperature, top-k and top-p, checked on the problem sheet's Q6.
- **KV cache**: generate with cached keys and values, and compare the result and the time with recomputing everything.

### **3.1 Training a Tiny GPT**

We train the `GPT` of Section 1.3 on the first chapter of Jane Austen's *Pride and Prejudice* (1813, public domain), about 4,400 characters, embedded in the cell below. The tokens are single characters, so the vocabulary is just the distinct characters of the text.

The model has $D = 128$, 4 heads, 2 blocks and a context length of 128, about 0.4M parameters. Each training step takes 16 random windows of 128 characters; the targets are the same windows shifted by one character, so the model learns to predict the next character at every position, always given the true previous characters. The loss is the cross-entropy, and the optimiser is Adam (a variant of gradient descent), for 800 steps, with a learning rate that decreases from $2 \times 10^{-3}$ to 0 along a cosine curve.

The model uses your `MultiHeadSelfAttention` if it passed the check of Exercise 1.2, and PyTorch's otherwise. The cell prints a greedy generation (always the most likely next character) before and after training, and plots the loss. The generation starts from "Mr. Bennet" rather than from the opening words "It is a truth": a random window contains the first characters of the text only if it starts exactly there, so the model sees them in only a handful of the 12,800 training windows, whereas every character from the middle of the text appears in a few hundred. Training takes less than a minute on a laptop CPU, depending on the machine.

In [ ]:
TEXT = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.
However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered the rightful property of some one or other of their daughters.
"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"
Mr. Bennet replied that he had not.
"But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."
Mr. Bennet made no answer.
"Do you not want to know who has taken it?" cried his wife impatiently.
"You want to tell me, and I have no objection to hearing it."
This was invitation enough.
"Why, my dear, you must know, Mrs. Long says that Netherfield is taken by a young man of large fortune from the north of England; that he came down on Monday in a chaise and four to see the place, and was so much delighted with it, that he agreed with Mr. Morris immediately; that he is to take possession before Michaelmas, and some of his servants are to be in the house by the end of next week."
"What is his name?"
"Bingley."
"Is he married or single?"
"Oh! Single, my dear, to be sure! A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"
"How so? How can it affect them?"
"My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."
"Is that his design in settling here?"
"Design! Nonsense, how can you talk so! But it is very likely that he may fall in love with one of them, and therefore you must visit him as soon as he comes."
"I see no occasion for that. You and the girls may go, or you may send them by themselves, which perhaps will be still better, for as you are as handsome as any of them, Mr. Bingley may like you the best of the party."
"My dear, you flatter me. I certainly have had my share of beauty, but I do not pretend to be anything extraordinary now. When a woman has five grown-up daughters, she ought to give over thinking of her own beauty."
"In such cases, a woman has not often much beauty to think of."
"But, my dear, you must indeed go and see Mr. Bingley when he comes into the neighbourhood."
"It is more than I engage for, I assure you."
"But consider your daughters. Only think what an establishment it would be for one of them. Sir William and Lady Lucas are determined to go, merely on that account, for in general, you know, they visit no newcomers. Indeed you must go, for it will be impossible for us to visit him if you do not."
"You are over-scrupulous, surely. I dare say Mr. Bingley will be very glad to see you; and I will send a few lines by you to assure him of my hearty consent to his marrying whichever he chooses of the girls; though I must throw in a good word for my little Lizzy."
"I desire you will do no such thing. Lizzy is not a bit better than the others; and I am sure she is not half so handsome as Jane, nor half so good-humoured as Lydia. But you are always giving her the preference."
"They have none of them much to recommend them," replied he; "they are all silly and ignorant like other girls; but Lizzy has something more of quickness than her sisters."
"Mr. Bennet, how can you abuse your own children in such a way? You take delight in vexing me. You have no compassion for my poor nerves."
"You mistake me, my dear. I have a high respect for your nerves. They are my old friends. I have heard you mention them with consideration these last twenty years at least."
"Ah, you do not know what I suffer."
"But I hope you will get over it, and live to see many young men of four thousand a year come into the neighbourhood."
"It will be no use to us, if twenty such should come, since you will not visit them."
"Depend upon it, my dear, that when there are twenty, I will visit them all."
Mr. Bennet was so odd a mixture of quick parts, sarcastic humour, reserve, and caprice, that the experience of three-and-twenty years had been insufficient to make his wife understand his character. Her mind was less difficult to develop. She was a woman of mean understanding, little information, and uncertain temper. When she was discontented, she fancied herself nervous. The business of her life was to get her daughters married; its solace was visiting and news.
"""

chars = sorted(set(TEXT))  # the vocabulary: every distinct character
stoi = {c: i for i, c in enumerate(chars)}


def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(chars[i] for i in ids)


data = torch.tensor(encode(TEXT))
print(f"{len(TEXT):,} characters, vocabulary of {len(chars)} characters: {''.join(chars)!r}")
print(f"'Mr. Bennet' -> {encode('Mr. Bennet')}")


def get_batch(data, batch_size, length, generator):
    """batch_size random windows of the text: inputs x (B, length) and targets y, the same windows shifted by one."""
    starts = torch.randint(0, len(data) - length, (batch_size,), generator=generator)
    x = torch.stack([data[s : s + length] for s in starts])
    y = torch.stack([data[s + 1 : s + length + 1] for s in starts])
    return x.to(device), y.to(device)


@torch.no_grad()
def generate(model, idx, n_new, next_token=None):
    """Append n_new tokens to idx (B, L), running the model on the whole sequence at every step (no cache).
    next_token maps the logits of the last position (B, V) to the next tokens (B, 1); greedy (argmax) by default."""
    for _ in range(n_new):
        logits = model(idx[:, -model.context_length :])[:, -1]  # (B, V): the prediction for the next token
        nxt = logits.argmax(dim=-1, keepdim=True) if next_token is None else next_token(logits)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

In [ ]:
import time

import matplotlib.pyplot as plt
import torch.nn.functional as F

attention_class = MultiHeadSelfAttention if mhsa_ok else ReferenceMultiHeadSelfAttention
print("Using " + ("your MultiHeadSelfAttention." if mhsa_ok else
      "PyTorch's multi-head attention until your MultiHeadSelfAttention passes Exercise 1.2 (then re-run this cell)."))

torch.manual_seed(SEED)
model = GPT(len(chars), d_model=128, n_heads=4, n_blocks=2, context_length=128, attention_class=attention_class).to(device)
print(f"Parameters: {count_parameters(model):,}")
PROMPT = "Mr. Bennet"
prompt_ids = torch.tensor([encode(PROMPT)], device=device)
print(f"\nbefore training (greedy): {decode(generate(model.eval(), prompt_ids, 80)[0].tolist())!r}\n")

STEPS = 800
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STEPS)  # learning rate from 2e-3 down to 0
g = torch.Generator().manual_seed(SEED)
print(f"random guessing: loss ln {len(chars)} = {math.log(len(chars)):.3f}")
losses = []
model.train()
start = time.time()
for step in range(1, STEPS + 1):
    x, y = get_batch(data, 16, 128, g)
    logits = model(x)  # (B, L, V): a prediction at every position
    loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    losses.append(loss.item())
    if step == 1 or step % 100 == 0:
        print(f"step {step:3d} | loss {loss.item():.3f} | {time.time() - start:.1f} s")
model.eval()

print(f"\nafter training (greedy):\n{decode(generate(model, prompt_ids, 200)[0].tolist())}")

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, len(losses) + 1), losses)
plt.axhline(math.log(len(chars)), color="gray", linestyle="--", label="random guessing")
plt.xlabel("training step")
plt.ylabel("cross-entropy loss")
plt.title("Tiny GPT on Pride and Prejudice, chapter 1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In our run, the loss starts at 3.97, close to random guessing ($\ln 51 \approx 3.93$), stays above 2 for about 200 steps, and then drops quickly, to about 0.11 after 800 steps. Before training, the greedy generation repeats a few characters; after training, it continues "Mr. Bennet" with the last paragraph of the chapter, word for word. A loss this low on only 4,400 characters means that the model has memorised most of the text (it has about 100 times more parameters than the text has characters), so this shows that training works, not that the model can write new text. Training took a few seconds on the Apple-silicon GPU (MPS) of a laptop and about 12 s on its CPU in our tests. The exact losses and the generated text can differ between machines and PyTorch versions.

### **3.2 Exercise: Temperature, Top-k and Top-p**

Instead of always taking the most likely token, we can sample the next token from the model's distribution, reshaped in one of three ways:

- **Temperature** $T$: $p = \text{softmax}(z / T)$ for the logits $z$; $T < 1$ sharpens the distribution and $T > 1$ flattens it.
- **Top-k**: keep the $k$ most likely tokens, set the other probabilities to 0, and renormalise.
- **Top-p**: keep the smallest set of most likely tokens whose cumulative probability exceeds $p$, and renormalise.

**Requirements**: complete `sample_probs(logits, temperature, top_k, top_p)`, which returns the distribution to sample from, for `logits` of shape `(..., V)`. Apply the temperature first, then top-k, then top-p; `None` means "not used".

1. **Temperature (TODO 1)**: `probs`, over the last dimension.
2. **Top-k (TODO 2)**: zero all but the `top_k` largest probabilities, and renormalise.
3. **Top-p (TODO 3)**: in decreasing order of probability, keep a token if the cumulative probability of the tokens *before* it is below `top_p`; zero the others, in the original order, and renormalise.

The verification code checks the problem sheet's Q6 (logits $[\log 8, \log 4, \log 2, 0, 0]$ for tokens a-e), and top-p once more with the tokens in another order.

**Hint**: useful are `torch.topk`, `torch.sort(..., descending=True)`, `torch.cumsum`, `torch.where`, and `scatter` with the indices returned by `sort` to go back to the original order.

In [ ]:
def sample_probs(logits, temperature=1.0, top_k=None, top_p=None):
    """The distribution (..., V) to sample the next token from, given logits (..., V)."""
    # TODO 1: temperature: probs = softmax(logits / temperature) over the last dimension
    probs = None
    if probs is None:  # TODO 1 not done yet
        return None

    if top_k is not None:
        # TODO 2: keep the top_k largest probabilities, set the others to 0, and renormalise
        pass

    if top_p is not None:
        # TODO 3: sort in decreasing order; keep a token if the cumulative probability of the tokens before it
        #         is below top_p; set the others to 0 (in the original order) and renormalise
        pass
    return probs


# --- Verification Code (Implemented) ---
print("-" * 30)
z = torch.log(torch.tensor([8.0, 4, 2, 1, 1]))  # the problem sheet's Q6, tokens a, b, c, d, e
cases = [
    ("(A) T = 1", z, {}, [1 / 2, 1 / 4, 1 / 8, 1 / 16, 1 / 16]),
    ("(B) T = 1/2", z, {"temperature": 0.5}, [32 / 43, 8 / 43, 2 / 43, 1 / 86, 1 / 86]),
    ("(C) top-k, k = 2", z, {"top_k": 2}, [2 / 3, 1 / 3, 0, 0, 0]),
    ("(D) top-p, p = 0.8", z, {"top_p": 0.8}, [4 / 7, 2 / 7, 1 / 7, 0, 0]),
    ("(D) reordered c, a, e, b, d", z[[2, 0, 4, 1, 3]], {"top_p": 0.8}, [1 / 7, 4 / 7, 0, 2 / 7, 0]),
]
sample_probs_ok = sample_probs(z) is not None
if not sample_probs_ok:
    print("Please complete the TODO sections to run the verification.")
for name, logits, options, expected in cases if sample_probs_ok else []:
    mine = sample_probs(logits.to(device), **options).cpu()
    expected = torch.tensor(expected)
    match = torch.allclose(mine, expected, atol=1e-6)  # float32 rounding of the logs; mistakes differ by 0.01 or more
    sample_probs_ok = sample_probs_ok and match
    print(f"Q6 {name:28s} yours {mine.numpy().round(4)}, expected {expected.numpy().round(4)}, match: {match}")

The cell below generates 200 characters with several settings, from the same prompt and the same random seed. It uses your `sample_probs` once it passes the check above; until then it applies only the temperature and ignores top-k and top-p. Try other settings.

In [ ]:
def next_token_sampler(temperature=1.0, top_k=None, top_p=None):
    """A function that samples the next tokens (B, 1) from the last logits (B, V)."""
    def sample(logits):
        if sample_probs_ok:
            probs = sample_probs(logits, temperature, top_k, top_p)
        else:  # temperature only
            probs = torch.softmax(logits / temperature, dim=-1)
        return torch.multinomial(probs, num_samples=1)
    return sample


if not sample_probs_ok:
    print("Your sample_probs does not pass the check yet: applying only the temperature.\n")
settings = [
    {"temperature": 0.5},
    {"temperature": 1.0},
    {"temperature": 2.0},
    {"temperature": 1.0, "top_k": 5},
    {"temperature": 1.0, "top_p": 0.8},
]
for options in settings:
    torch.manual_seed(SEED)
    out = generate(model, prompt_ids, 200, next_token=next_token_sampler(**options))
    print(f"--- {options}\n{decode(out[0].tolist())}\n")

In our run, the sample at $T = 0.5$ reproduces the last paragraph of the chapter word for word, like greedy decoding. At $T = 1$, an occasional unlikely character sends the model off track, and the sample mixes words of the chapter with invented ones; at $T = 2$ it is close to random characters, including rare capitals and punctuation. Top-p ($p = 0.8$) removes the unlikely characters and gives a passage of the chapter almost word for word. Top-k ($k = 5$) always keeps five characters, even where the model is almost certain of one, so its sample still contains invented words. Your samples will differ from ours on another machine.

### **3.3 Exercise: Generating with a KV Cache**

`generate` in Section 3.1 runs the model on the whole sequence for every new token. With the causal mask, the keys and values of earlier tokens never change, so they can be stored and reused:

1. **Prefill**: run the model once on the whole prompt and store the keys and values of every block. The prompt tokens must not see later prompt tokens, so the causal mask is needed here.
2. **Decoding**: run the model on the new token only. Every block appends the token's key and value to its cache, and the token's query attends to all cached keys; as the last token, it needs no mask.

**Requirements**:

- In `attention_with_cache(attn, x, cache)`, for the new tokens `x` of shape `(B, L_new, D_k)`, the per-head `Q`, `K` and `V` of shape `(B, h, L_new, D_h)` are provided; `cache` is an empty dict before the prefill.
  1. **Cache (TODO 1)**: store `K` and `V` as `cache["K"]` and `cache["V"]`, or append them along the sequence dimension if the cache is not empty.
  2. **Attention (TODO 2)**: the new queries over all cached keys and values, with `attention(...)` from Exercise 1.1 and the causal mask only in the prefill.
- In `generate_with_cache(model, idx, n_new)`:
  3. **Prefill (TODO 3)**: `forward_with_cache` on the whole prompt.
  4. **Decoding (TODO 4)**: `forward_with_cache` on the new token only.

`forward_with_cache` (provided) runs the whole model on the new tokens and gives them the positions after the cached ones. The verification code checks that greedy generation with the cache gives exactly the same characters as `generate`, and then times both.

**Hint**: `torch.cat(..., dim=2)` appends along the sequence dimension of a `(B, h, L, D_h)` tensor.

In [ ]:
import copy
import time


def split_heads(t, h):
    """(B, L, D_k) -> (B, h, L, D_h)."""
    B, L, D_k = t.shape
    return t.view(B, L, h, D_k // h).transpose(1, 2)


def merge_heads(t):
    """(B, h, L, D_h) -> (B, L, D_k)."""
    B, h, L, D_h = t.shape
    return t.transpose(1, 2).reshape(B, L, h * D_h)


def attention_with_cache(attn, x, cache):
    """Multi-head attention of the new tokens x (B, L_new, D_k) over all tokens so far; updates cache in place.
    cache is {} before the prefill and holds "K" and "V" of shape (B, h, L_so_far, D_h) afterwards."""
    Q, K, V = (split_heads(layer(x), attn.n_heads) for layer in (attn.W_q, attn.W_k, attn.W_v))  # (B, h, L_new, D_h)

    # TODO 1: store K and V in cache["K"] and cache["V"] if the cache is empty; otherwise append them along the sequence dimension
    pass

    # TODO 2: attention of the new queries Q over all cached keys and values, with the causal mask only in the prefill
    #         (more than one new token); shape (B, h, L_new, D_h)
    O = None

    if O is None:  # TODO not done yet
        return None
    return attn.W_o(merge_heads(O))


def forward_with_cache(model, idx, caches):
    """Logits (B, L_new, V) for the new tokens idx (B, L_new), given the caches of earlier tokens (one dict per block)."""
    n_cached = caches[0]["K"].shape[2] if caches[0] else 0
    assert n_cached + idx.shape[1] <= model.context_length, "the sequence is longer than the context length"
    positions = torch.arange(n_cached, n_cached + idx.shape[1], device=idx.device)  # the new tokens come after the cached ones
    x = model.tok_emb(idx) + model.pos_emb(positions)
    for block, cache in zip(model.blocks, caches):
        a = attention_with_cache(block.attn, block.ln1(x), cache)
        if a is None:  # TODO 1 or TODO 2 not done yet
            return None
        x = x + a
        x = x + block.mlp(block.ln2(x))
    return model.head(model.ln_f(x))


@torch.no_grad()
def generate_with_cache(model, idx, n_new):
    """Greedy generation of n_new tokens after idx (B, L) with a KV cache."""
    caches = [{} for _ in model.blocks]  # one empty cache per block

    # TODO 3: prefill: run forward_with_cache on the whole prompt idx
    logits = None

    for _ in range(n_new):
        if logits is None:  # TODO not done yet
            return None
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # (B, 1), greedy
        idx = torch.cat([idx, next_token], dim=1)

        # TODO 4: run forward_with_cache on the new token only, reusing the caches
        logits = None
    return idx


# --- Verification Code (Implemented) ---
print("-" * 30)
N_NEW = model.context_length - len(PROMPT)  # the prompt and the new characters fill the context length of 128
reference = generate(model, prompt_ids, N_NEW)
mine = generate_with_cache(model, prompt_ids, N_NEW)
if mine is None:
    print("Please complete the TODO sections to run the verification.")
else:
    n_same = (mine[0] == reference[0]).sum().item() if mine.shape == reference.shape else 0
    print(f"with cache: {decode(mine[0, len(PROMPT):len(PROMPT) + 60].tolist())!r} ...")
    print(f"no cache:   {decode(reference[0, len(PROMPT):len(PROMPT) + 60].tolist())!r} ...")
    print(f"identical tokens: {n_same} of {reference.shape[1]}, match: {torch.equal(mine, reference)}\n")


    def timed(fn, dev):
        """fn() and its wall-clock time in seconds, waiting for the GPU to finish."""
        sync = torch.cuda.synchronize if dev.type == "cuda" else torch.mps.synchronize if dev.type == "mps" else (lambda: None)
        sync()
        start = time.perf_counter()
        out = fn()
        sync()
        return out, time.perf_counter() - start


    # Timing: the same architecture with a context length of 1024, untrained (the time does not depend on the weights)
    N_TIME = 500
    torch.manual_seed(SEED)
    long_model = GPT(len(chars), d_model=128, n_heads=4, n_blocks=2, context_length=1024, attention_class=attention_class).eval()
    for dev in [torch.device("cpu")] + ([device] if device.type != "cpu" else []):
        m, p = long_model.to(dev), prompt_ids.to(dev)
        generate(m, p, 5), generate_with_cache(m, p, 5)  # warm-up
        _, t_plain = timed(lambda: generate(m, p, N_TIME), dev)
        _, t_cache = timed(lambda: generate_with_cache(m, p, N_TIME), dev)
        print(f"{dev.type}: {N_TIME} new tokens without cache {t_plain:.2f} s, with cache {t_cache:.2f} s "
              f"({t_plain / t_cache:.1f} times faster)")

In our run, greedy generation with and without the cache gave the same 128 characters. On a laptop CPU, generating 500 tokens was several times faster with the cache (about 8 to 9 times in repeated runs); on the laptop's Apple-silicon GPU (MPS), the cache helped much less (about 1.3 times), and both were slower than on the CPU, because a model this small is limited by the overhead of launching many small GPU operations rather than by arithmetic. Your timings will differ, but the pattern should be similar. Without a cache, step $t$ processes all $t$ tokens again, so the total work grows with the square of the length; the saving therefore grows with the sequence length and the model size. The price is memory: the cache holds the keys and values of every token in every block (the problem sheet's Q3 C).

## Part 4: LoRA (Optional)

- **A LoRA layer**: wrap a linear layer with a frozen weight and a trainable low-rank update, check it on the problem sheet's Q7, and fine-tune the tiny GPT of Section 3.1 on a new text by training only the LoRA parameters.

### **4.1 Exercise: A LoRA Layer**

LoRA freezes a pretrained weight $W$ and trains only a low-rank update (lecture p.83):

$$h = Wx + BAx, \qquad A \in \mathbb{R}^{r \times d},\ A_{ij} \sim \mathcal{N}(0, \sigma^2), \qquad B \in \mathbb{R}^{d \times r},\ B = 0 \text{ at the start}.$$

**Scenario**:
`LoRALinear(base, r, sigma)` wraps an `nn.Linear` layer `base` with input size `d_in` and output size `d_out`, so $A$ is $r \times d_\text{in}$ and $B$ is $d_\text{out} \times r$. The rows of `x` are tokens, so $BAx$ for every row is `x @ A.T @ B.T`. The base layer keeps its bias, also frozen.

**Requirements**:

1. **Freeze (TODO 1)**: no gradients for the base layer's weight and bias.
2. **Parameters (TODO 2)**: `self.A` drawn from $\mathcal{N}(0, \sigma^2)$ and `self.B` filled with zeros, both `nn.Parameter`.
3. **Forward (TODO 3)**: `base(x)` plus $BAx$ for every row of `x`.

The verification code checks that the output equals the base layer's output before training, that the base layer is frozen, and the number of trainable parameters of the problem sheet's Q7. The next cell fine-tunes the tiny GPT with your layer.

**Hint**: `requires_grad_(False)` freezes all parameters of a module.

In [ ]:
import copy
import math

import torch.nn as nn


class LoRALinear(nn.Module):
    """h = W x + B A x (lecture p.83): the frozen layer `base` plus a trainable update B A of rank r."""

    def __init__(self, base, r, sigma=None):
        super().__init__()
        self.base = base
        if sigma is None:
            sigma = 1 / math.sqrt(base.in_features)  # then A x has entries of about the size of those of x

        # TODO 1: freeze the base layer's weight and bias
        pass

        # TODO 2: self.A (r x d_in) with entries from N(0, sigma^2) and self.B (d_out x r) of zeros, both nn.Parameter
        self.A = self.B = None

    def forward(self, x):
        # TODO 3: base(x) + B A x for every row of x
        return None


# --- Verification Code (Implemented) ---
print("-" * 30)
torch.manual_seed(SEED)
base = nn.Linear(64, 64).to(device)
x = torch.randn(5, 64, device=device)
lora = LoRALinear(copy.deepcopy(base), r=4).to(device)
out = lora(x) if lora.A is not None else None
lora_ok = out is not None
if not lora_ok:
    print("Please complete the TODO sections to run the verification.")
else:
    same = torch.equal(out, base(x))  # B = 0, so B A x is exactly 0
    out.sum().backward()
    frozen = lora.base.weight.grad is None and lora.base.bias.grad is None and lora.B.grad is not None
    print(f"before training, LoRA output = base output: {same}")
    print(f"base layer frozen, A and B trainable: {frozen}")

    with torch.device("meta"):  # shapes only, no memory
        big = LoRALinear(nn.Linear(4096, 4096), r=8)
    trainable = sum(p.numel() for p in big.parameters() if p.requires_grad)
    d, r = 4096, 8
    print(f"Q7: trainable parameters for d = {d}, r = {r}: {trainable:,} (2dr = {2 * d * r:,}), match: {trainable == 2 * d * r}")
    print(f"Q7: W_q and W_v in 32 blocks: {64 * trainable:,} trainable parameters; fraction of one matrix: "
          f"{trainable / d**2} = 1/{d**2 // trainable}")
    lora_ok = same and frozen and trainable == 2 * d * r

Now we fine-tune a copy of the tiny GPT of Section 3.1 on a new text, the opening sentence of Charles Dickens's *A Tale of Two Cities* (1859, public domain). As in the problem sheet's Q7, we wrap $W_q$ and $W_v$ of every block with your `LoRALinear` (rank $r = 16$) and train only $A$ and $B$; everything else stays frozen. The cell prints the generation from the prompt "It was the" before and after fine-tuning. It needs Exercise 4.1 to pass its check.

In [ ]:
import copy

import torch.nn.functional as F

NEW_TEXT = """It was the best of times, it was the worst of times, it was the age of wisdom, it was the age of foolishness, it was the epoch of belief, it was the epoch of incredulity, it was the season of Light, it was the season of Darkness, it was the spring of hope, it was the winter of despair, we had everything before us, we had nothing before us, we were all going direct to Heaven, we were all going direct the other way--in short, the period was so far like the present period, that some of its noisiest authorities insisted on its being received, for good or for evil, in the superlative degree of comparison only.
"""
new_data = torch.tensor(encode(NEW_TEXT))  # every character of NEW_TEXT also occurs in TEXT

if not lora_ok:
    print("Please complete Exercise 4.1 first.")
else:
    torch.manual_seed(SEED)
    lora_model = copy.deepcopy(model)
    lora_model.requires_grad_(False)  # freeze the whole pretrained model
    for block in lora_model.blocks:
        block.attn.W_q = LoRALinear(block.attn.W_q, r=16)
        block.attn.W_v = LoRALinear(block.attn.W_v, r=16)
    lora_model.to(device)
    trainable = [p for p in lora_model.parameters() if p.requires_grad]
    print(f"trainable parameters: {sum(p.numel() for p in trainable):,} of {count_parameters(lora_model):,}")

    new_prompt = torch.tensor([encode("It was the")], device=device)
    print(f"\nbefore fine-tuning: {decode(generate(lora_model, new_prompt, 120)[0].tolist())!r}\n")

    optimizer = torch.optim.Adam(trainable, lr=3e-3)
    g = torch.Generator().manual_seed(SEED)
    lora_model.train()
    start = time.time()
    for step in range(1, 501):
        x, y = get_batch(new_data, 16, 128, g)
        logits = lora_model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step == 1 or step % 100 == 0:
            print(f"step {step:3d} | loss {loss.item():.3f} | {time.time() - start:.1f} s")
    lora_model.eval()
    print(f"\nafter fine-tuning:  {decode(generate(lora_model, new_prompt, 120)[0].tolist())!r}")
    print(f"original model:     {decode(generate(model, new_prompt, 120)[0].tolist())!r}")

In our run, only 16,384 of the 436,096 parameters (about 4 %) were trained. Before fine-tuning, the model continues "It was the" with pieces of the Austen chapter; after 500 steps, it reproduces part of the Dickens sentence, while the original model is unchanged. The fine-tuning loss fell much faster than in Section 3.1, partly because the frozen model already knows the characters and many common words. The generated text can differ between machines.

## Summary

- Causal attention is $\text{softmax}(QK^\top/\sqrt{d_k})\,V$ with the scores above the diagonal set to $-\infty$, so that no token attends to later tokens. Multi-head attention reshapes `(B, L, D_k)` to `(B, h, L, D_h)`, runs all heads at once, and merges them before $W_o$.
- One GPT-2 block has $12D^2 + 13D$ parameters, and GPT-2 small has 124,439,808 with weight tying. For 1,024 tokens, the attention scores of one layer already take about 25 MB, more than the 20 MB of SRAM.
- RoPE rotates each pair of entries of the queries and keys by $m\theta_i$, so the attention score depends only on the distance $n - m$.
- Attention computed block by block, with a running maximum and a running denominator, gives exactly the same result as ordinary attention while keeping only one block of scores at a time.
- Temperature, top-k and top-p reshape the next-token distribution. A KV cache gives the same tokens as recomputing everything, while each decoding step processes only the new token.
- LoRA trains $2dr$ parameters per $d \times d$ matrix; since $B = 0$ at the start, the adapted model starts out exactly as the pretrained one.